In [4]:
import os
import json
import fitz
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document


print("All libraries loaded!")

All libraries loaded!


C:\Users\azbas\AppData\Local\Temp\ipykernel_15164\869555949.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [5]:
### pdf parsing

def extract_text_from_pdf(pdf_path):
    """Extract text from PDF using PyMuPDF"""
    doc = fitz.open(pdf_path)
    text = ""
    for page_num in range(len(doc)):
        page = doc[page_num]
        text += page.get_text()
    doc.close()
    return text

def chunk_text(text, chunk_size=512, overlap=100):
    """Split text into chunks"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    
    return chunks

def process_all_pdfs():
    """Process all PDFs in data/reports/"""
    reports_dir = "data/reports"
    
    if not os.path.exists(reports_dir):
        print(f"Error: {reports_dir} folder not found!")
        return
    
    all_chunks = []
    
    for pdf_file in os.listdir(reports_dir):
        if pdf_file.endswith('.pdf'):
            pdf_path = os.path.join(reports_dir, pdf_file)
            company_name = pdf_file.replace('.pdf', '').replace('2025_2026', '')
            
            print(f"Processing: {pdf_file}...")
            
            text = extract_text_from_pdf(pdf_path)
            
            chunks = chunk_text(text)
            
            for i, chunk in enumerate(chunks):
                chunk_data = {
                    "company": company_name,
                    "pdf_file": pdf_file,
                    "chunk_id": i,
                    "text": chunk,
                    "tokens": len(chunk.split())
                }
                all_chunks.append(chunk_data)
            
            print(f"  → Created {len(chunks)} chunks from {company_name}")
    
    # Save chunks to JSON
    output_file = "data/chunks.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(all_chunks, f, indent=2, ensure_ascii=False)
    
    print(f"\nTotal chunks created: {len(all_chunks)}")
    print(f"Chunks saved to: {output_file}")
    return all_chunks

chunks = process_all_pdfs()

Processing: ADANIPOWER2025_2026.pdf...
  → Created 621 chunks from ADANIPOWER
Processing: AMBUJACEMENT2025_2026.pdf...
  → Created 700 chunks from AMBUJACEMENT
Processing: ASIANPAINTS2025_2026.pdf...
  → Created 576 chunks from ASIANPAINTS
Processing: BAJAJAUTO2025_2026.pdf...
  → Created 421 chunks from BAJAJAUTO
Processing: CIPLA2025_2026.pdf...
  → Created 563 chunks from CIPLA
Processing: COLGATE2025_2026.pdf...
  → Created 301 chunks from COLGATE
Processing: INFOSYS2025_2026.pdf...
  → Created 471 chunks from INFOSYS
Processing: ITC2025_2026.pdf...
  → Created 488 chunks from ITC
Processing: JSWSTEEL2025_2026.pdf...
  → Created 870 chunks from JSWSTEEL
Processing: M&M2025_2026.pdf...
  → Created 580 chunks from M&M
Processing: RELIANCE2025_2026.pdf...
  → Created 416 chunks from RELIANCE
Processing: SUNPHARMA2025_2026.pdf...
  → Created 391 chunks from SUNPHARMA
Processing: TATAPOWER2025_2026.pdf...
  → Created 742 chunks from TATAPOWER
Processing: TATASTEEL2025_2026.pdf...
  → Cr

In [6]:
# Load chunks from JSON
with open('data/chunks.json', 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

print(f"Loaded {len(chunks_data)} chunks")
print(f"First chunk text sample: {chunks_data[0]['text'][:100]}...")

Loaded 10675 chunks
First chunk text sample: Adani Power Limited Tel +91 79 2656 7555 “Adani Corporate House” Fax +91 79 2555 7177 Shantigram, Ne...


In [7]:
### Embedding


print("Loading Hugging Face Sentence Transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding all chunks (this will take a few minutes)...")
texts = [chunk['text'] for chunk in chunks_data]
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

print(f"Embeddings created: {embeddings.shape}")

# Add embeddings to chunks
for i, chunk in enumerate(chunks_data):
    chunk['embedding'] = embeddings[i].tolist()

print(f"Total chunks with embeddings: {len(chunks_data)}")

Loading Hugging Face Sentence Transformer model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7518.24it/s]


Embedding all chunks (this will take a few minutes)...


Batches:  15%|█▌        | 51/334 [01:11<06:36,  1.40s/it]


KeyboardInterrupt: 

In [10]:
### store data in chromadb


documents = []
for chunk in chunks_data:
    doc = Document(
        page_content=chunk['text'],
        metadata={
            'company': chunk['company'],
            'pdf_file': chunk['pdf_file'],
            'chunk_id': chunk['chunk_id']
        }
    )
    documents.append(doc)

print(f"Created {len(documents)} documents for Chroma")

print("Initializing Chroma vector database...")
embeddings_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings_model,
    persist_directory="./chroma_db"
)

print(f"Chroma vector database created!")
print(f"Stored {len(documents)} documents in Chroma")

Created 10675 documents for Chroma
Initializing Chroma vector database...


C:\Users\azbas\AppData\Local\Temp\ipykernel_15164\1999448248.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3788.23it/s]


Chroma vector database created!
Stored 10675 documents in Chroma


In [ ]:
# Test retrieval
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a Retrieval-Augmented Financial Analysis Assistant.

You are NOT a general AI assistant.

Your ONLY source of truth is the retrieved context provided to you.

====================================================
NON-NEGOTIABLE RULES
====================================================

1. NEVER answer using your own knowledge.

2. NEVER use information that is not explicitly present in the retrieved context.

3. NEVER guess.

4. NEVER infer missing values.

5. NEVER estimate numbers.

6. NEVER complete partial information.

7. NEVER assume anything.

8. If the answer is not fully supported by the retrieved context, respond EXACTLY with:

9. maximum 5 sentence answer that is highly relevant.

"The retrieved documents do not contain sufficient information to answer this question."

Do not provide any additional explanation.

====================================================
NUMERICAL DATA
====================================================

If the question involves:

- Revenue
- Profit
- EPS
- EBITDA
- Assets
- Liabilities
- Debt
- Ratios
- Percentages
- Shareholding
- Cash Flow
- Financial Statements

then:

- Copy numbers exactly.
- Preserve units exactly.
- Never round.
- Never calculate unless the document explicitly provides the calculation.
- Never derive missing values.

====================================================
COMPARISON QUESTIONS
====================================================

Only compare information that exists in the retrieved context.

If one company is missing, respond with:

"The retrieved documents do not contain sufficient information to perform this comparison."

====================================================
WHY QUESTIONS
====================================================

If asked:

"Why?"

"What caused?"

"What was the reason?"

Only answer if the retrieved text explicitly states the reason.

Otherwise respond:

"The retrieved documents do not specify the reason."

====================================================
SUMMARIZATION
====================================================

Summarize ONLY retrieved information.

Do not add background knowledge.

Do not explain industry concepts.

Do not provide interpretations.

====================================================
MULTIPLE SOURCES
====================================================

If retrieved chunks disagree:

- State the conflict.
- Quote both pieces of evidence.
- Do not decide which is correct.

====================================================
OUTPUT FORMAT
====================================================

Answer:
<answer>

Evidence:

For every factual statement include:

- Company:
- PDF:
- Chunk ID:
- Exact supporting quotation (maximum 40 words)

Confidence:

HIGH
Only if every statement is directly supported.

MEDIUM
If only part of the answer is supported.

LOW
If evidence is weak.

If confidence would be LOW, instead return:

"The retrieved documents do not contain sufficient information to answer this question."

====================================================
ABSOLUTE RULE
====================================================

If there is any uncertainty whatsoever,

DO NOT ANSWER.

Return:

"The retrieved documents do not contain sufficient information to answer this question."

Context:
{context}

Question:
{question}

Context:
{context}

Question:
{question}

Answer:
"""
)

print("Testing retrieval...\n")

query = "What is the company's revenue?"
results = vectorstore.similarity_search(query, k=5)

print(f"Retrieved {len(results)} chunks for: '{query}'\n")

for i, result in enumerate(results, 1):
    print(f"Result {i}: {result.metadata['company'].upper()}")
    print(f"  Text: {result.page_content[:100]}...\n")

print("Testing retrieval...\n")
query = "What is the company's revenue?"
results = vectorstore.similarity_search(query, k=5)

print(f"Retrieved {len(results)} chunks for: '{query}'\n")
for i, result in enumerate(results, 1):
    print(f"Result {i}: {result.metadata['company'].upper()}")
    print(f"  Text: {result.page_content[:100]}...\n")

Testing retrieval...

Retrieved 5 chunks for: 'What is the company's revenue?'

Result 1: INFOSYS2025_2026
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 2: INFOSYS2025_2026
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 3: INFOSYS
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 4: INFOSYS
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 5: INFOSYS
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Testing retrieval...

Retrieved 5 chunks for: 'What is the company's revenue?'

Result 1: INFOSYS2025_2026
  Text: Group presents revenues net of indirect taxes in its Consolidated Statement of Profit and Loss. Reve...

Result 2: INFOSYS2025_2026


In [15]:
# trying to connect Grok to the retrieval system
# get question -> retrieve chunks -> prepare for Grok

import os
from dotenv import load_dotenv

load_dotenv()

# check if grok key is there
grok_key = os.getenv('GROK_API_KEY')
print(f"Grok key: {grok_key[:10]}..." if grok_key else "No grok key found!")

# function to retrieve and prepare context
def prepare_for_grok(question):
    # retrieve chunks
    results = vectorstore.similarity_search(question, k=5)
    
    # combine into context
    context = "\n".join([result.page_content for result in results])
    
    print(f"Q: {question}")
    print(f"Retrieved {len(results)} chunks")
    print(f"\nContext prepared for Grok:")
    print(context[:300])
    
    return context, results

print("Ready to send to Grok")

Grok key: xai-w5xloP...
Ready to send to Grok


In [18]:
context, docs = prepare_for_grok("What is adani revenue?")

Q: What is adani revenue?
Retrieved 5 chunks

Context prepared for Grok:
Enterprises Limited 532.87 - 532.87 - 15 Loans interest - Repaid Adani Enterprises Limited 20.50 - 20.50 - 719 718 Integrated Annual Report 2025-26 AMBUJA CEMENTS LIMITED Portfolio Overview Corporate Overview Strategic Review ESG Overview Statutory Reports Financial Statements Notes to Consolidated 


In [ ]:
import sys
print(sys.executable)

In [6]:
print(f"Total chunks: {len(chunks_data)}")
print(f"Unique chunks:{len(set(chunk['text'] for chunk in chunks_data))}")

Total chunks: 10675
Unique chunks:10675


In [19]:
context, docs = prepare_for_grok("total revenue sales financial performance")
print(context[:500])


Q: total revenue sales financial performance
Retrieved 5 chunks

Context prepared for Grok:
as at the end of the reporting period and an explanation as to when the Group expects to recognise these amounts in revenue. Applying the practical expedient as given in Ind AS 115, the Group has not disclosed the remaining performance obligation related disclosures for contracts as the revenue reco
as at the end of the reporting period and an explanation as to when the Group expects to recognise these amounts in revenue. Applying the practical expedient as given in Ind AS 115, the Group has not disclosed the remaining performance obligation related disclosures for contracts as the revenue recognised corresponds directly with the value to the customer of the entity’s performance completed to date. The aggregate value of performance obligations that are partially unsatisfied as at March 31, 


In [20]:
context, docs = prepare_for_grok("Reliance profit margin")
print(context[:200])

Q: Reliance profit margin
Retrieved 5 chunks

Context prepared for Grok:
actual payment basis post Ind AS implementation w.e.f. FY 2016-17 ^ For Reliance Industries Limited * Pursuant to issue of Bonus Shares in 2017-18 and 2024-25 in the ratio of 1:1 ** Refer Note 40.1 of Standalone Financial Statements Note: Above highlights are part of Management Discussion and Analys
actual payment basis post Ind AS implementation w.e.f. FY 2016-17 ^ For Reliance Industries Limited * Pursuant to issue of Bonus Shares in 2017-18 and 2024-25 in the ratio of 1:1 ** Refer Note 40.1 of


In [21]:
context, docs = prepare_for_grok("financial statements net profit earnings")
print(context[:200])

Q: financial statements net profit earnings
Retrieved 5 chunks

Context prepared for Grok:
value through OCI (2,413) 1,189 Income taxes relating to items that will be reclassified to profit or loss 28 2,576 34 Total other comprehensive income for the year, net of taxes 35,522 4,148 Total comprehensive income for the year 168,177 136,328 Profit for the year attributable to: Equity holders 
value through OCI (2,413) 1,189 Income taxes relating to items that will be reclassified to profit or loss 28 2,576 34 Total other comprehensive income for the year, net of taxes 35,522 4,148 Total co


In [22]:
# Check for duplicate chunks in results

context, docs = prepare_for_grok("financial statements net profit earnings")

print(f"Retrieved {len(docs)} chunks")
print("\nChecking for duplicates:")

for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}:")
    print(f"Company: {doc.metadata['company']}")
    print(f"Text: {doc.page_content[:80]}...")

# See if any are the same

Q: financial statements net profit earnings
Retrieved 5 chunks

Context prepared for Grok:
value through OCI (2,413) 1,189 Income taxes relating to items that will be reclassified to profit or loss 28 2,576 34 Total other comprehensive income for the year, net of taxes 35,522 4,148 Total comprehensive income for the year 168,177 136,328 Profit for the year attributable to: Equity holders 
Retrieved 5 chunks

Checking for duplicates:

Chunk 1:
Company: WIPRO2025_2026
Text: value through OCI (2,413) 1,189 Income taxes relating to items that will be recl...

Chunk 2:
Company: WIPRO
Text: value through OCI (2,413) 1,189 Income taxes relating to items that will be recl...

Chunk 3:
Company: WIPRO
Text: value through OCI (2,413) 1,189 Income taxes relating to items that will be recl...

Chunk 4:
Company: WIPRO
Text: 38,202 K 36,491 Foreign exchange gains/(losses), net on financial instruments me...

Chunk 5:
Company: WIPRO
Text: 38,202 K 36,491 Foreign exchange gains/(losses), net on financi

In [ ]:
# Remove duplicate chunks from retrieval

def prepare_for_grok_no_duplicates(question):
    # retrieve chunks
    results = vectorstore.similarity_search(question, k=10)  # get more to account for duplicates
    
    # remove duplicates
    seen = set()
    unique_results = []
    
    for result in results:
        text_hash = hash(result.page_content[:100]) 
        
        if text_hash not in seen:
            seen.add(text_hash)
            unique_results.append(result)
    
    # combine into context
    context = "\n".join([result.page_content for result in unique_results])
    
    print(f"Q: {question}")
    print(f"Retrieved {len(results)} chunks, {len(unique_results)} unique")
    print(f"\nContext prepared:")
    print(context[:400])
    
    return context, unique_results

print("Ready - duplicates removed")

Ready - duplicates removed


In [24]:
context, docs = prepare_for_grok_no_duplicates("financial statements net profit earnings")

Q: financial statements net profit earnings
Retrieved 10 chunks, 4 unique

Context prepared:
value through OCI (2,413) 1,189 Income taxes relating to items that will be reclassified to profit or loss 28 2,576 34 Total other comprehensive income for the year, net of taxes 35,522 4,148 Total comprehensive income for the year 168,177 136,328 Profit for the year attributable to: Equity holders of the Company 131,974 131,354 Non-controlling interests 681 826 132,655 132,180 Other comprehensive


In [8]:
print("vectorstore" in globals())

False


In [9]:
collection = vectorstore._collection

print("Documents in Chroma:", collection.count())
print("Documents in your code:", len(documents))

NameError: name 'vectorstore' is not defined